In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import json

import pandas as pd

from src.modules import get_hf_tokenizer_embeddings

c:\Users\Nizwa\miniconda3\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## News Dataset Overview

In [2]:
NEWS_COLUMNS = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]

news_df = pd.read_csv(
    "../.dataset/train/news.tsv",
    sep="\t",
    header=None,
    names=NEWS_COLUMNS,
    dtype=str,
)
news_df.head()

,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N88753,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N45436,news,newsscienceandtechnology,Walmart Slashes Prices on Last-Generation iPads,Apple's new iPad releases bring big deals on l...,https://assets.msn.com/labs/mind/AABmf2I.html,"[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ...","[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ..."
2,N23144,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
3,N86255,health,medical,Dispose of unwanted prescription drugs during ...,NaN,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[]
4,N93187,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."


In [3]:
news_df.info()
news_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101527 entries, 0 to 101526
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   news_id            101527 non-null  object
 1   category           101527 non-null  object
 2   subcategory        101527 non-null  object
 3   title              101527 non-null  object
 4   abstract           96112 non-null   object
 5   url                101527 non-null  object
 6   title_entities     101524 non-null  object
 7   abstract_entities  101521 non-null  object
dtypes: object(8)
memory usage: 6.2+ MB


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
count,101527,101527,101527,101527,96112,101527,101524,101521
unique,101527,18,285,98388,91654,101526,66863,72168
top,N56840,sports,newsus,Powerball Winning Numbers For 10/26/2019 Drawi...,What's the weather today? What's the weather f...,[],[],[]
freq,1,32020,14467,27,435,2,26053,25480


### Category and Sub-Category data

In [4]:
unique_category = news_df["category"].unique()
unique_category, len(unique_category)

(array(['lifestyle', 'news', 'health', 'sports', 'weather',
        'entertainment', 'foodanddrink', 'autos', 'travel', 'video',
        'finance', 'tv', 'movies', 'music', 'kids', 'middleeast', 'games',
        'northamerica'], dtype=object),
 18)

In [5]:
unique_subcategory = news_df["subcategory"].unique()
unique_subcategory, len(unique_subcategory)

(array(['lifestyleroyals', 'newsscienceandtechnology', 'weightloss',
        'medical', 'newsworld', 'voices', 'cardio', 'football_nfl',
        'weathertopstories', 'gaming', 'recipes', 'lifestylelovesex',
        'nutrition', 'autosenthusiasts', 'autossports', 'wellness',
        'health-news', 'celebrity', 'travelarticle', 'autossuvs',
        'newspolitics', 'more_sports', 'traveltripideas', 'animals',
        'autosnews', 'newsbusiness', 'golf', 'newstrends',
        'lifestylepetsanimals', 'finance-insurance', 'football_ncaa',
        'lifestylebuzz', 'mma', 'fitness', 'newsus', 'tv-gallery',
        'tvnews', 'lifestylehoroscope', 'basketball_nba', 'news',
        'shop-all', 'newsphotos', 'lifestylemindandsoul', 'travelnews',
        'basketball_ncaa', 'finance-real-estate', 'quickandeasy',
        'tv-celebrity', 'financenews', 'lifestyleparenting',
        'movies-gallery', 'racing', 'tipsandtricks', 'baseball_mlb',
        'musicnews', 'autosbuying', 'shop-apparel', 'autostr

### Number of entities across news

In [6]:
def count_entities(row):
    entities = []

    for column in ["title_entities", "abstract_entities"]:
        try:
            data = json.loads(row[column])
        except (json.JSONDecodeError, TypeError):
            data = []

        entities.extend(
            entity["WikidataId"]
            for entity in data
            if entity.get("WikidataId")
        )

    # same behavior as your NewsDatabase
    return len(dict.fromkeys(entities))


news_df["entity_count"] = news_df.apply(count_entities, axis=1)

news_df["entity_count"].describe()

count    101527.000000
mean          2.453968
std           1.883541
min           0.000000
25%           1.000000
50%           2.000000
75%           3.000000
max          30.000000
Name: entity_count, dtype: float64

In [7]:
news_df["entity_count"].value_counts().sort_index()

entity_count
0     11887
1     23804
2     23899
3     17579
4     10944
5      6430
6      3520
7      1799
8       817
9       423
10      201
11       96
12       53
13       27
14       19
15        8
16        7
17        6
18        2
19        2
20        1
22        1
24        1
30        1
Name: count, dtype: int64

In [8]:
print("the count shows that a news with 0 entity, has about 11,000 data, and so on")
print("meaning having a max entities at either 5 to 10 entities feed into the model, is a good threshold")

the count shows that a news with 0 entity, has about 11,000 data, and so on
meaning having a max entities at either 5 to 10 entities feed into the model, is a good threshold


### Other split (validation & test)

In [9]:
val_news_df = pd.read_csv(
    "../.dataset/validation/news.tsv",
    sep="\t",
    header=None,
    names=NEWS_COLUMNS,
    dtype=str,
)
val_news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72023 entries, 0 to 72022
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   news_id            72023 non-null  object
 1   category           72023 non-null  object
 2   subcategory        72023 non-null  object
 3   title              72023 non-null  object
 4   abstract           68400 non-null  object
 5   url                72023 non-null  object
 6   title_entities     72021 non-null  object
 7   abstract_entities  72018 non-null  object
dtypes: object(8)
memory usage: 4.4+ MB


In [10]:
test_news_df = pd.read_csv(
    "../.dataset/test/news.tsv",
    sep="\t",
    header=None,
    names=NEWS_COLUMNS,
    dtype=str,
)
test_news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120959 entries, 0 to 120958
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   news_id            120959 non-null  object
 1   category           120959 non-null  object
 2   subcategory        120959 non-null  object
 3   title              120959 non-null  object
 4   abstract           114256 non-null  object
 5   url                120958 non-null  object
 6   title_entities     120953 non-null  object
 7   abstract_entities  120950 non-null  object
dtypes: object(8)
memory usage: 7.4+ MB


### Truncation thershold by token ids length percentile

In [11]:
# 2 common option here:
# BERT = "bert-base-uncased"
# MiniLM = "microsoft/MiniLM-L12-H384-uncased"

_, _, tokenizer = get_hf_tokenizer_embeddings(model_name="microsoft/MiniLM-L12-H384-uncased") 
# make sure the model_name matches the actual training config

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 66274.93it/s]


In [12]:
news_df["title"] = news_df["title"].fillna("")
news_df["abstract"] = news_df["abstract"].fillna("")


# Tokenize WITHOUT truncation/padding
title_tokens = tokenizer(
    news_df["title"].tolist(),
    add_special_tokens=True,
    truncation=False,
    padding=False,
)

abstract_tokens = tokenizer(
    news_df["abstract"].tolist(),
    add_special_tokens=True,
    truncation=False,
    padding=False,
)

news_df["title_token_length"] = [
    len(x) for x in title_tokens["input_ids"]
]

news_df["abstract_token_length"] = [
    len(x) for x in abstract_tokens["input_ids"]
]

In [13]:
percentiles = [50, 75, 80, 85, 90, 95, 97, 98, 99, 99.5, 99.9, 100]

print("TITLE")
print(
    news_df["title_token_length"]
    .describe(percentiles=[p / 100 for p in percentiles])
)

print("\nABSTRACT")
print(
    news_df["abstract_token_length"]
    .describe(percentiles=[p / 100 for p in percentiles])
)

TITLE
count    101527.000000
mean         15.805185
std           4.632011
min           4.000000
50%          15.000000
75%          18.000000
80%          19.000000
85%          20.000000
90%          22.000000
95%          24.000000
97%          26.000000
98%          27.000000
99%          29.000000
99.5%        32.000000
99.9%        38.000000
100%        132.000000
max         132.000000
Name: title_token_length, dtype: float64

ABSTRACT
count    101527.000000
mean         49.187674
std          36.100173
min           2.000000
50%          34.000000
75%          87.000000
80%          92.000000
85%          96.000000
90%         100.000000
95%         106.000000
97%         110.000000
98%         113.000000
99%         118.000000
99.5%       128.000000
99.9%       207.474000
100%        626.000000
max         626.000000
Name: abstract_token_length, dtype: float64


In [14]:
print("max title length can be set to 40")
print("max abstract length can be set to 130")

max title length can be set to 40
max abstract length can be set to 130


## Behavior Dataset Overview

In [15]:
BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

behavior_df = pd.read_csv(
    "../.dataset/train/behaviors.tsv",
    sep="\t",
    header=None,
    names=BEHAVIOR_COLUMNS,
    dtype=str,
)
behavior_df.head()

,impression_id,user_id,time,history,impressions
0,1,U87243,11/10/2019 11:30:54 AM,N8668 N39081 N65259 N79529 N73408 N43615 N2937...,N78206-0 N26368-0 N7578-0 N58592-0 N19858-0 N5...
1,2,U598644,11/12/2019 1:45:29 PM,N56056 N8726 N70353 N67998 N83823 N111108 N107...,N47996-0 N82719-0 N117066-0 N8491-0 N123784-0 ...
2,3,U532401,11/13/2019 11:23:03 AM,N128643 N87446 N122948 N9375 N82348 N129412 N5...,N103852-0 N53474-0 N127836-0 N47925-1
3,4,U593596,11/12/2019 12:24:09 PM,N31043 N39592 N4104 N8223 N114581 N92747 N1207...,N38902-0 N76434-0 N71593-0 N100073-0 N108736-0...
4,5,U239687,11/14/2019 8:03:01 PM,N65250 N122359 N71723 N53796 N41663 N41484 N11...,N76209-0 N48841-0 N67937-0 N62235-0 N6307-0 N3...


In [16]:
test_behavior_df = pd.read_csv(
    "../.dataset/test/behaviors.tsv",
    sep="\t",
    header=None,
    names=BEHAVIOR_COLUMNS,
    dtype=str,
)

test_behavior_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2370727 entries, 0 to 2370726
Data columns (total 5 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   impression_id  object
 1   user_id        object
 2   time           object
 3   history        object
 4   impressions    object
dtypes: object(5)
memory usage: 90.4+ MB


In [23]:
test_behavior_df.head()

,impression_id,user_id,time,history,impressions
0,1,U64099,11/19/2019 11:37:45 AM,N121133 N104200 N43255 N55860 N128965 N38014 N...,N101071 N15647 N83400 N124838 N57092 N64623 N6...
1,2,U231077,11/19/2019 5:28:08 AM,N45124 N84730 N45128 N104312 N70022 N99111 N26...,N14657 N51253 N49521 N126571 N74286 N101071 N1...
2,3,U606012,11/19/2019 4:46:23 AM,N59893 N84662 N90686 N33265 N127225 N120859 N6...,N74286 N9250 N26898 N123737 N98301 N80580 N456...
3,4,U320649,11/21/2019 6:03:51 AM,N110863 N7889 N86335 N85056 N115743 N63372 N19...,N119559 N37657 N108085 N91287 N39136 N130190 N...
4,5,U357840,11/22/2019 10:36:19 AM,N98596 N85005 N15713 N67779 N47961 N55571 N666...,N60658 N43496 N65220 N9125 N63136 N83728 N3208...


In [17]:
behavior_df["history"].isna().sum()

np.int64(46065)

In [18]:
behavior_df["history"] = (
    behavior_df["history"]
    .fillna("")
)

In [19]:
behavior_df.info()
behavior_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2232748 entries, 0 to 2232747
Data columns (total 5 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   impression_id  object
 1   user_id        object
 2   time           object
 3   history        object
 4   impressions    object
dtypes: object(5)
memory usage: 85.2+ MB


,impression_id,user_id,time,history,impressions
count,2232748,2232748,2232748,2232748,2232748
unique,2232748,711222,453122,689270,1972081
top,2232748,U536528,11/11/2019 11:21:20 AM,,N98178-1 N32154-0
freq,1,106,29,46065,6130


In [20]:
behavior_df["history_count"] = (
    behavior_df["history"]
    .fillna("")
    .str.split()
    .str.len()
)

behavior_df["impression_count"] = (
    behavior_df["impressions"]
    .str.split()
    .str.len()
)

behavior_df["positive_count"] = (
    behavior_df["impressions"]
    .str.findall(r"-1(?:\s|$)")
    .str.len()
)

behavior_df["negative_count"] = (
    behavior_df["impressions"]
    .str.findall(r"-0(?:\s|$)")
    .str.len()
)

In [21]:
pd.set_option("display.float_format", "{:.2f}".format)

behavior_df[
    [
        "history_count",
        "impression_count",
        "positive_count",
        "negative_count",
    ]
].describe()

,history_count,impression_count,positive_count,negative_count
count,2232748.00,2232748.00,2232748.00,2232748.00
mean,32.98,37.40,1.52,35.89
std,40.87,38.74,1.17,38.29
min,0.00,2.00,1.00,1.00
25%,8.00,10.00,1.00,9.00
50%,19.00,25.00,1.00,23.00
75%,42.00,51.00,2.00,50.00
max,801.00,300.00,51.00,299.00


In [22]:
print("about user max history to be proceed by the model:")
print("set max history to 50, because at 75% percentile, the history reaches the count of 42, so either 40-50 max history is a good boundary")
print()

print("about model training objectives:")
print("we are trying to predict the user impression, each candidate news impression has label 0 and 1, where 0 is ignore and 1 is click (predicting click probability)")
print("therefore we're gonna be using the cross entropy loss, but cross entropy only proceed 1 positive sample for its primary objective and minimize the rest N negative sample")
print("but the dataset has many positive sample, so the strategy is to split impression where each row must contains only 1 positive label, and having rest negative")
print("therefore we need to decide how many negative samples are assign as candidates, more negative candidates means more compute")
print("at the 75% percentile, the negative count reaches 50, and having 50 negative candidate is super expensive")
print("so we can pick any low number, eg: 4 (like most paper did) and we can shuffle the negative sample randomly for each epochs")
print("so total candidate is = 1 positive + 4 negative = 5 candidate news")
print()

print("about dataset size:")
print("as we discussed, 1 positive label = 1 training example, meaning the data grow so much bigger the more positive label that exist in the dataset")
print("shown by the data, we have about 2,232,748 impression count, and an positive count of 1.5, hence the multiplier can result roughly over 3 million rows")
print("for a smaller size but efficient, the percentile shown in 50% most data are still having 1 positive label, while 75% having 2 label")
print("here we can process all data that has a positive label of 1, so here we can filter out more data and prevent growing data size if having a hardware concern")

about user max history to be proceed by the model:
set max history to 50, because at 75% percentile, the history reaches the count of 42, so either 40-50 max history is a good boundary

about model training objectives:
we are trying to predict the user impression, each candidate news impression has label 0 and 1, where 0 is ignore and 1 is click (predicting click probability)
therefore we're gonna be using the cross entropy loss, but cross entropy only proceed 1 positive sample for its primary objective and minimize the rest N negative sample
but the dataset has many positive sample, so the strategy is to split impression where each row must contains only 1 positive label, and having rest negative
therefore we need to decide how many negative samples are assign as candidates, more negative candidates means more compute
at the 75% percentile, the negative count reaches 50, and having 50 negative candidate is super expensive
so we can pick any low number, eg: 4 (like most paper did) and 